# Lesson 1: Connect to Aura Graph Analytics

**Module:** AGA Fundamentals  
**Dataset:** Movies (Actors, Movies, Genres, Users)

This notebook confirms your environment is correctly set up. You'll authenticate against the Aura API using the credentials in your `.env` file, create a small session attached to your AuraDB instance, and verify the connection works end to end.

If every cell runs without errors, you're ready to move on to `2_syntax.ipynb`.

## Load your credentials

The cell below pulls your `.env` credentials into local variables, ready to use across the rest of the lesson. The imports bring in everything you'll need from `graphdatascience.session`:

* `GdsSessions`: the manager that creates and tears down AGA sessions
* `AuraAPICredentials`: wraps the Client ID and Secret used to talk to the Aura API
* `DbmsConnectionInfo`: describes which AuraDB instance a session should attach to
* `SessionMemory`: the memory-tier enum used when sizing a session
* `timedelta` (from the standard library): sets the session's time-to-live

In [1]:
import os
from datetime import timedelta
from graphdatascience.session import GdsSessions, AuraAPICredentials, DbmsConnectionInfo, SessionMemory
from dotenv import load_dotenv

load_dotenv()

client_id     = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id    = os.getenv('AURA_PROJECT_ID')
uri           = os.getenv('AURA_URI')
username      = os.getenv('AURA_USERNAME')
password      = os.getenv('AURA_PASSWORD')

print(f"Client ID starts with: {client_id[:12]}..." if client_id else "AURA_CLIENT_ID is not set")

Client ID starts with: izGLQbUSdYjU...


/Users/t6w652j6ft/Documents/GitHub/aga-fundamentals-notebooks/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The print line should show the first few characters of your Client ID, confirming the credentials are loaded and ready to use.

## Build the sessions manager

To run algorithms, you'll first need to project graphs into a session. To create a session, you need a session manager.

The code below creates a `GdsSessions` manager and hands it your Aura API credentials so it can talk to the Aura control plane on your behalf. From here on, everything session-related — creating, listing, attaching, deleting — flows through this `sessions` object.

In [2]:
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id)
)
print("Sessions manager ready.")

Sessions manager ready.


You now have a `sessions` manager ready to create or attach to AGA sessions.

## Spin up a session

Next, you'll use the `sessions` manager to spin up an AGA session connected to your AuraDB instance. The session is where algorithms actually run — once it's live, you can project graphs into it from the database.

For this first session you'll go small: the smallest memory tier (`m_2GB`) and a 15-minute TTL, enough to confirm the full credentials-and-connection chain works.

In [3]:
gds = sessions.get_or_create(
    session_name="aga-fundamentals-connect-check",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password,
    ),
    ttl=timedelta(minutes=15),
)
gds.verify_connectivity()
print("Connected to GDS Session: aga-fundamentals-connect-check")

Connected to GDS Session: aga-fundamentals-connect-check


If you see `Connected to GDS Session` with no exception, your environment is correctly configured. If you see an authentication error, double-check your secrets — particularly that `AURA_CLIENT_ID` and `AURA_CLIENT_SECRET` haven't been swapped.

## Clean up

To end the session explicitly, run `sessions.delete`. The meter stops the moment the session terminates.

In [4]:
sessions.delete(session_name="aga-fundamentals-connect-check")
print("Session deleted.")

Session deleted.


Even if you leave the session running, it will expire after the time set for `ttl`.

Move on to `2_syntax.ipynb` to run the full AGA workflow end to end.